# MCP Architecture, Lifecycle, and Trust Boundaries

## Scenario and safety boundary
A tenant-scoped support host connects to a ticket server. This notebook is offline: it uses a deterministic protocol-shaped fixture and makes no network call or side effect.

**Success criterion:** explain every lifecycle message and identify the security owner at each boundary.

## Mental model
The host selects integrations and owns user approval; one client manages each protocol session; the server validates its own inputs and downstream authority. Capability discovery is untrusted metadata, not permission.

In [ ]:
import runpy
from pathlib import Path
module = runpy.run_path(Path('lab.py'))
trace = module['build_trace']()
trace[:3]

## Baseline: initialize, negotiate, confirm
Inspect the first three messages. A request has an `id`; a notification does not. The server's capabilities are claims to review before use.

In [ ]:
for event in trace:
    print(event.get('id', 'notification'), event.get('method', 'response'))

## Experiment: typed tool boundary
The trace permits one ticket identifier. A production server must validate this independently of any model decision.

In [ ]:
review = module['security_review']
assert review(trace) == []
bad = [dict(event) for event in trace]
bad[6] = {**bad[6], 'params': {'name': 'ticket.read', 'arguments': {'ticket_id': 'acme-7', 'debug': True}}}
review(bad)

## Failure injection and mitigation
Removing `notifications/initialized` models a client that calls methods before it has completed its negotiation protocol. The fixture reports this as an observable invariant failure.

In [ ]:
incomplete = [event for event in trace if event.get('method') != 'notifications/initialized']
review(incomplete)

## Evaluation and production upgrade
A trace is reviewable when it joins server identity/version, session and trace IDs, user/tenant, capability, policy decision, destination, result class, and revocation owner without logging raw tokens. For a real server, replace this fixture with Course 04's official SDK implementation, add typed validation, correlation IDs, timeouts, and a tested revocation path.

## Reflection
Why cannot a `tools/list` response authorize `tools/call`? Identify the separate host, server, and downstream enforcement points.